In [1]:
import pandas as pd
import os 

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------
csv_file = "iterative_improvement_results.csv"
df = pd.read_csv(csv_file)

# Expected columns:
# instance, pivoting_rule, neighborhood, initial_solution, cost, delta_percent, time_seconds


In [3]:
# ------------------------------------------------------------
# Table 1: Main results for the 12 iterative improvement algorithms
# ------------------------------------------------------------
table1 = (
    df.groupby(["pivoting_rule", "neighborhood", "initial_solution"], as_index=False)
      .agg(
          avg_deviation_percent=("delta_percent", "mean"),
          std_deviation_percent=("delta_percent", "std"),
          total_time_seconds=("time_seconds", "sum"),
          avg_time_per_instance_seconds=("time_seconds", "mean")
      )
)

table1 = table1.sort_values(
    by=["avg_deviation_percent", "total_time_seconds"],
    ascending=[True, True]
).reset_index(drop=True)

print("\n=== TABLE 1: Main results for the 12 algorithms ===")
# print(table1.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
table1


=== TABLE 1: Main results for the 12 algorithms ===


,pivoting_rule,neighborhood,initial_solution,avg_deviation_percent,std_deviation_percent,total_time_seconds,avg_time_per_instance_seconds
0,best,exchange,cw,16.428363,1.476654,9.528577,0.122161
1,best,insert,cw,16.640161,1.510425,19.687603,0.252405
2,best,transpose,cw,17.267723,1.613588,0.334854,0.004293
3,first,exchange,cw,17.274370,1.611134,0.228910,0.002935
4,first,insert,cw,17.274450,1.612997,0.354285,0.004542
5,first,transpose,cw,17.283998,1.614963,0.207048,0.002654
6,best,exchange,random,33.213704,3.430176,8.950350,0.114748
7,best,insert,random,33.912177,3.682344,18.467649,0.236765
8,best,transpose,random,34.983818,4.209485,0.110000,0.001410
9,first,exchange,random,35.211865,4.264636,0.001878,0.000024


In [4]:
# ------------------------------------------------------------
# Table 2: Best algorithms by quality and by speed
# ------------------------------------------------------------
best_quality = table1.loc[table1["avg_deviation_percent"].idxmin()]
second_best_quality = table1.nsmallest(2, "avg_deviation_percent").iloc[1]
fastest = table1.loc[table1["total_time_seconds"].idxmin()]
slowest = table1.loc[table1["total_time_seconds"].idxmax()]

cw_only = table1[table1["initial_solution"] == "cw"]
fastest_cw = cw_only.loc[cw_only["total_time_seconds"].idxmin()]

table2 = pd.DataFrame([
    {
        "criterion": "Best average deviation",
        "pivoting_rule": best_quality["pivoting_rule"],
        "neighborhood": best_quality["neighborhood"],
        "initial_solution": best_quality["initial_solution"],
        "value": best_quality["avg_deviation_percent"]
    },
    {
        "criterion": "Second best average deviation",
        "pivoting_rule": second_best_quality["pivoting_rule"],
        "neighborhood": second_best_quality["neighborhood"],
        "initial_solution": second_best_quality["initial_solution"],
        "value": second_best_quality["avg_deviation_percent"]
    },
    {
        "criterion": "Fastest total time",
        "pivoting_rule": fastest["pivoting_rule"],
        "neighborhood": fastest["neighborhood"],
        "initial_solution": fastest["initial_solution"],
        "value": fastest["total_time_seconds"]
    },
    {
        "criterion": "Fastest CW-based algorithm",
        "pivoting_rule": fastest_cw["pivoting_rule"],
        "neighborhood": fastest_cw["neighborhood"],
        "initial_solution": fastest_cw["initial_solution"],
        "value": fastest_cw["total_time_seconds"]
    },
    {
        "criterion": "Slowest algorithm",
        "pivoting_rule": slowest["pivoting_rule"],
        "neighborhood": slowest["neighborhood"],
        "initial_solution": slowest["initial_solution"],
        "value": slowest["total_time_seconds"]
    }
])

print("\n=== TABLE 2: Best/worst algorithms ===")
# print(table2.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
table2 


=== TABLE 2: Best/worst algorithms ===


,criterion,pivoting_rule,neighborhood,initial_solution,value
0,Best average deviation,best,exchange,cw,16.428363
1,Second best average deviation,best,insert,cw,16.640161
2,Fastest total time,first,exchange,random,0.001878
3,Fastest CW-based algorithm,first,transpose,cw,0.207048
4,Slowest algorithm,best,insert,cw,19.687603


In [11]:
# ------------------------------------------------------------
# Table 3: Effect of the initial solution
# ------------------------------------------------------------

table3 = (
    table1.groupby("initial_solution", as_index=False)
          .agg(mean_of_avg_deviations=("avg_deviation_percent", "mean"))
)

print("\n=== TABLE 3: Effect of the initial solution ===")
# print(table3.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
table3
if set(table3["initial_solution"]) == {"cw", "random"}:
    cw_mean = table3.loc[table3["initial_solution"] == "cw", "mean_of_avg_deviations"].iloc[0]
    rand_mean = table3.loc[table3["initial_solution"] == "random", "mean_of_avg_deviations"].iloc[0]
    print(f"\nCW reduces the mean deviation by {rand_mean - cw_mean:.6f} percentage points.")
table3



=== TABLE 3: Effect of the initial solution ===

CW reduces the mean deviation by 17.596878 percentage points.


,initial_solution,mean_of_avg_deviations
0,cw,17.028177
1,random,34.625056


In [12]:
# ------------------------------------------------------------
# Table 4: Effect of pivoting rule within same initialization and neighborhood
# ------------------------------------------------------------
table4 = (
    table1.pivot_table(
        index=["initial_solution", "neighborhood"],
        columns="pivoting_rule",
        values="avg_deviation_percent"
    )
    .reset_index()
)

if "first" in table4.columns and "best" in table4.columns:
    table4["better_one"] = table4.apply(
        lambda row: "best" if row["best"] < row["first"] else ("first" if row["first"] < row["best"] else "tie"),
        axis=1
    )

print("\n=== TABLE 4: First vs Best within each neighborhood and initialization ===")
# print(table4.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
table4 


=== TABLE 4: First vs Best within each neighborhood and initialization ===


pivoting_rule,initial_solution,neighborhood,best,first,better_one
0,cw,exchange,16.428363,17.274370,best
1,cw,insert,16.640161,17.274450,best
2,cw,transpose,17.267723,17.283998,best
3,random,exchange,33.213704,35.211865,best
4,random,insert,33.912177,35.212548,best
5,random,transpose,34.983818,35.216223,best


In [14]:
# ------------------------------------------------------------
# Table 5: Effect of neighborhood within same pivoting rule and initialization
# ------------------------------------------------------------
table5 = (
    table1.pivot_table(
        index=["pivoting_rule", "initial_solution"],
        columns="neighborhood",
        values="avg_deviation_percent"
    )
    .reset_index()
)

def best_neighborhood(row):
    candidates = {}
    for n in ["transpose", "exchange", "insert"]:
        if n in row.index and pd.notna(row[n]):
            candidates[n] = row[n]
    if not candidates:
        return None
    best_val = min(candidates.values())
    best_names = [k for k, v in candidates.items() if v == best_val]
    return " / ".join(best_names)

table5["best_neighborhood"] = table5.apply(best_neighborhood, axis=1)

print("\n=== TABLE 5: Neighborhood comparison within pivoting rule and initialization ===")
table5



=== TABLE 5: Neighborhood comparison within pivoting rule and initialization ===


neighborhood,pivoting_rule,initial_solution,exchange,insert,transpose,best_neighborhood
0,best,cw,16.428363,16.640161,17.267723,exchange
1,best,random,33.213704,33.912177,34.983818,exchange
2,first,cw,17.274370,17.274450,17.283998,exchange
3,first,random,35.211865,35.212548,35.216223,exchange


In [21]:
# ADDING BEST KNOWN VALUE TO iterative_improvement_results.csv to help comparison 


# ------------------------------------------------------------
# Files
# ------------------------------------------------------------
INPUT_RESULTS = "iterative_improvement_results.csv"
BEST_KNOWN_FILE = "../best_known/best_known.txt"
OUTPUT_FILE = "iterative_improvement_results_with_best.csv"

# ------------------------------------------------------------
# Load results CSV
# ------------------------------------------------------------
df = pd.read_csv(INPUT_RESULTS)

# Check instance column
if "instance" not in df.columns:
    raise ValueError("Column 'instance' not found in CSV")

# ------------------------------------------------------------
# Load best_known.txt
# ------------------------------------------------------------
best_known_dict = {}

with open(BEST_KNOWN_FILE, "r") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 2:
            instance_name = parts[0].strip()
            value = int(parts[1])
            best_known_dict[instance_name] = value

# Debug print
print(f"Loaded {len(best_known_dict)} best-known values")

# ------------------------------------------------------------
# Match instance names
# ------------------------------------------------------------
def extract_instance_name(full_path):
    """
    If your CSV has full paths like:
    ./instances/N-be75eec_150.dat
    → extract N-be75eec_150
    """
    name = full_path.split("/")[-1]      # remove path
    name = name.replace(".dat", "")      # remove extension
    return name

# Apply extraction
df["instance_clean"] = df["instance"].apply(extract_instance_name)

# Map best_known values
df["best_known"] = df["instance_clean"].map(best_known_dict)

# ------------------------------------------------------------
# Check for missing matches
# ------------------------------------------------------------
missing = df[df["best_known"].isna()]["instance_clean"].unique()

if len(missing) > 0:
    print("\n WARNING: Missing best-known values for:")
    for m in missing:
        print(" -", m)
else:
    print("\nAll instances matched successfully!")


df = df[["instance", "pivoting_rule", "neighborhood", "initial_solution","best_known" ,"cost", "delta_percent", "time_seconds"]]
df

Loaded 78 best-known values

All instances matched successfully!


,instance,pivoting_rule,neighborhood,initial_solution,best_known,cost,delta_percent,time_seconds
0,N-tiw56r67_250,first,transpose,random,5289658,3716700,29.736478,0.000087
1,N-tiw56r67_250,first,transpose,cw,5289658,4415193,16.531598,0.006715
2,N-tiw56r67_250,first,exchange,random,5289658,3707395,29.912388,0.000044
3,N-tiw56r67_250,first,exchange,cw,5289658,4415322,16.529159,0.008832
4,N-tiw56r67_250,first,insert,random,5289658,3707418,29.911953,0.000050
...,...,...,...,...,...,...,...,...
931,N-be75tot_250,best,transpose,cw,30958609,25682978,17.040917,0.006758
932,N-be75tot_250,best,exchange,random,30958609,21413653,30.831346,0.201285
933,N-be75tot_250,best,exchange,cw,30958609,25876198,16.416794,0.251603
934,N-be75tot_250,best,insert,random,30958609,21263793,31.315412,0.431095


In [18]:
import pandas as pd
from itertools import combinations
from scipy.stats import ttest_rel, wilcoxon

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
INPUT_CSV = "iterative_improvement_results.csv"
OUTPUT_CSV = "pairwise_algorithm_comparisons.csv"
ALPHA = 0.05

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------
df = pd.read_csv(INPUT_CSV)

# Expected columns:
# instance, pivoting_rule, neighborhood, initial_solution, cost, delta_percent, time_seconds

required_cols = {
    "instance",
    "pivoting_rule",
    "neighborhood",
    "initial_solution",
    "delta_percent"
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ------------------------------------------------------------
# Build algorithm label
# ------------------------------------------------------------
df["algorithm"] = (
    df["pivoting_rule"].astype(str) + "_" +
    df["neighborhood"].astype(str) + "_" +
    df["initial_solution"].astype(str)
)

# ------------------------------------------------------------
# Get all algorithms
# ------------------------------------------------------------
algorithms = sorted(df["algorithm"].unique())
print(f"Found {len(algorithms)} algorithms:")
for a in algorithms:
    print(" -", a)

# ------------------------------------------------------------
# Pairwise comparison function
# ------------------------------------------------------------
def compare_two_algorithms(df_all, alg1, alg2):
    df1 = df_all[df_all["algorithm"] == alg1][["instance", "delta_percent"]].copy()
    df2 = df_all[df_all["algorithm"] == alg2][["instance", "delta_percent"]].copy()

    df1 = df1.rename(columns={"delta_percent": "delta_1"})
    df2 = df2.rename(columns={"delta_percent": "delta_2"})

    merged = pd.merge(df1, df2, on="instance", how="inner").sort_values("instance")

    if merged.empty:
        raise ValueError(f"No common instances between {alg1} and {alg2}")

    x = merged["delta_1"].to_numpy()
    y = merged["delta_2"].to_numpy()

    # Mean values
    mean_1 = x.mean()
    mean_2 = y.mean()
    mean_diff = (x - y).mean()   # positive => alg1 worse if lower delta is better
    abs_mean_diff = abs(mean_diff)

    # Lower delta_percent is better
    if mean_1 < mean_2:
        better_mean = alg1
    elif mean_2 < mean_1:
        better_mean = alg2
    else:
        better_mean = "tie"

    # Paired t-test
    t_stat, p_ttest = ttest_rel(x, y, nan_policy="omit")

    # Wilcoxon signed-rank test
    # If all differences are exactly zero, wilcoxon fails
    diffs = x - y
    if (diffs == 0).all():
        w_stat = 0.0
        p_wilcoxon = 1.0
    else:
        try:
            w_stat, p_wilcoxon = wilcoxon(x, y, zero_method="wilcox")
        except ValueError:
            # fallback in edge cases
            w_stat, p_wilcoxon = float("nan"), float("nan")

    return {
        "algorithm_1": alg1,
        "algorithm_2": alg2,
        "n_instances": len(merged),
        "mean_delta_1": mean_1,
        "mean_delta_2": mean_2,
        "mean_diff_delta_1_minus_2": mean_diff,
        "abs_mean_diff": abs_mean_diff,
        "better_by_mean": better_mean,
        "t_statistic": t_stat,
        "p_value_ttest": p_ttest,
        "significant_ttest": p_ttest < ALPHA if pd.notna(p_ttest) else False,
        "wilcoxon_statistic": w_stat,
        "p_value_wilcoxon": p_wilcoxon,
        "significant_wilcoxon": p_wilcoxon < ALPHA if pd.notna(p_wilcoxon) else False,
    }

# ------------------------------------------------------------
# Compare all pairs
# ------------------------------------------------------------
results = []

for alg1, alg2 in combinations(algorithms, 2):
    res = compare_two_algorithms(df, alg1, alg2)
    results.append(res)

results_df = pd.DataFrame(results)

# Sort: strongest Wilcoxon significance first, then t-test
results_df = results_df.sort_values(
    by=["p_value_wilcoxon", "p_value_ttest", "abs_mean_diff"],
    ascending=[True, True, False]
).reset_index(drop=True)

results_df

Found 12 algorithms:
 - best_exchange_cw
 - best_exchange_random
 - best_insert_cw
 - best_insert_random
 - best_transpose_cw
 - best_transpose_random
 - first_exchange_cw
 - first_exchange_random
 - first_insert_cw
 - first_insert_random
 - first_transpose_cw
 - first_transpose_random


,algorithm_1,algorithm_2,n_instances,mean_delta_1,mean_delta_2,mean_diff_delta_1_minus_2,abs_mean_diff,better_by_mean,t_statistic,p_value_ttest,significant_ttest,wilcoxon_statistic,p_value_wilcoxon,significant_wilcoxon
0,best_exchange_random,best_insert_cw,78,33.213704,16.640161,16.573543,16.573543,best_insert_cw,42.622212,2.629625e-55,True,0.0,1.682175e-14,True
1,best_exchange_cw,best_exchange_random,78,16.428363,33.213704,-16.785341,16.785341,best_exchange_cw,-42.603410,2.716792e-55,True,0.0,1.682175e-14,True
2,best_exchange_random,best_transpose_cw,78,33.213704,17.267723,15.945981,15.945981,best_transpose_cw,42.109985,6.423790e-55,True,0.0,1.682175e-14,True
3,best_exchange_random,first_exchange_cw,78,33.213704,17.274370,15.939335,15.939335,first_exchange_cw,42.095229,6.592186e-55,True,0.0,1.682175e-14,True
4,best_exchange_random,first_transpose_cw,78,33.213704,17.283998,15.929706,15.929706,first_transpose_cw,42.088123,6.674872e-55,True,0.0,1.682175e-14,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,first_exchange_random,first_transpose_random,78,35.211865,35.216223,-0.004359,0.004359,first_exchange_random,-0.320502,7.494558e-01,False,430.0,7.588163e-05,True
62,first_exchange_cw,first_transpose_cw,78,17.274370,17.283998,-0.009628,0.009628,first_exchange_cw,-2.470614,1.569886e-02,True,145.0,1.269132e-04,True
63,first_insert_cw,first_transpose_cw,78,17.274450,17.283998,-0.009548,0.009548,first_insert_cw,-2.274246,2.573604e-02,True,222.0,4.110033e-03,True
64,first_exchange_cw,first_insert_cw,78,17.274370,17.274450,-0.000080,0.000080,first_exchange_cw,-0.014719,9.882945e-01,False,330.0,1.287134e-01,False
